In [1]:
from dotenv import load_dotenv
load_dotenv('env')

True

In [2]:
import os
import re
import torch
from transformers import AutoTokenizer
from datasets import load_dataset
from tqdm.auto import tqdm

from io import StringIO
from contextlib import redirect_stdout
from time import perf_counter
from huggingface_hub import HfFileSystem, hf_hub_download
import jsonlines
from vllm import LLM, SamplingParams
import pandas as pd
from typing import Tuple, Optional
from collections import Counter

tqdm.pandas()

In [3]:
MODEL_ID = "Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256"
# SAVE_FPATH = "AIMO_Finetuned_Models/AIMO-SFT-DDP-gemma-7b-N141128-E1-R128-A64"

In [4]:
%%time
llm = LLM(
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    # kv_cache_dtype='fp8',
    dtype='float16',
    gpu_memory_utilization=0.9,
    tensor_parallel_size=torch.cuda.device_count(),
    # trust_remote_code=True,
    # adapter_name_or_path=LORA_ADAPTER_ID,
    swap_space=30,
    enable_prefix_caching=True,
    disable_log_stats=True,
    # enable_chunked_prefill=True,
    # max_num_batched_tokens=8192,
)

/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2024-06-11 17:54:00,040	INFO worker.py:1740 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


INFO 06-11 17:54:01 llm_engine.py:161] Initializing an LLM engine (v0.4.3) with config: model='Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256', speculative_config=None, tokenizer='Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=Hrushi/AIMO-SFT-DDP-Merged-Deepseek-Math-7B-Base-N141128-E1-R128-A256)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


INFO 06-11 17:54:09 utils.py:618] Found nccl from library libnccl.so.2
INFO 06-11 17:54:09 pynccl.py:65] vLLM is using nccl==2.20.5
(RayWorkerWrapper pid=29412) INFO 06-11 17:54:09 utils.py:618] Found nccl from library libnccl.so.2
(RayWorkerWrapper pid=29412) INFO 06-11 17:54:09 pynccl.py:65] vLLM is using nccl==2.20.5
INFO 06-11 17:54:09 custom_all_reduce_utils.py:179] reading GPU P2P access cache from /home/jupyter/.config/vllm/gpu_p2p_access_cache_for_0,1.json
(RayWorkerWrapper pid=29412) INFO 06-11 17:54:09 custom_all_reduce_utils.py:179] reading GPU P2P access cache from /home/jupyter/.config/vllm/gpu_p2p_access_cache_for_0,1.json
INFO 06-11 17:54:09 weight_utils.py:207] Using model weights format ['*.safetensors']
(RayWorkerWrapper pid=29412) INFO 06-11 17:54:09 weight_utils.py:207] Using model weights format ['*.safetensors']
INFO 06-11 17:54:11 model_runner.py:146] Loading model weights took 6.4663 GB
(RayWorkerWrapper pid=29412) INFO 06-11 17:54:12 model_runner.py:146] Loadin

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
def get_final_solution(x):
    sols = re.findall(r'\\boxed\{(.*?)\}', x, re.DOTALL)

    if len(sols) == 0:
        sols = re.findall(r'\\boxed (.*?)\$', x, re.DOTALL)
    
    # Check if all are same
    try:
        if all(sol == sols[0] for sol in sols):
            return sols[0]
        else:
            return sols[-1]
    except:
        print(sols)
        print(x)
        raise


In [7]:
# MATH_ds = load_dataset('hendrycks/competition_math', trust_remote_code=True)['test']
# MATH_df = MATH_ds.to_pandas()

# MATH_df['final_answer'] = MATH_df['solution'].progress_apply(get_final_solution)
# MATH_df['final_answer_is_numeric'] = MATH_df['final_answer'].progress_apply(lambda x: x.isnumeric())

# MATH_numeric_df = MATH_df[MATH_df['final_answer_is_numeric']].copy(deep=True).reset_index(drop=True)
# MATH_numeric_df['ExpID'] = [f"Exp{idx:0>4}" for idx in range(len(MATH_numeric_df))]
# MATH_numeric_df

df = pd.read_parquet('AIMO-Math-Problems.parquet')
df

,ExpID,Type,Problem,Solution,FinalAnswer
0,Exp00000,MATH_AnsAug,Gracie and Joe are choosing numbers on the com...,"The distance between two points $(x_1,y_1)$ an...",\sqrt{5}
1,Exp00001,GSM_Rephrased,What is the total cost of purchasing equipment...,"Each player requires a $25 jersey, a $15.20 pa...",752
2,Exp00002,GSM_SV,Diego baked 12 cakes for his sister's birthday...,"To solve this problem, we need to determine th...",1
3,Exp00003,MATH_AnsAug,Convert $10101_3$ to a base 10 integer.,$10101_3 = 1 \cdot 3^4 + 0 \cdot 3^3 + 1 \cdot...,91
4,Exp00004,GSM_FOBAR,"Sue works in a factory and every 30 minutes, a...","We know that every 30 minutes, a machine produ...",1
...,...,...,...,...,...
153611,Exp153611,Precalculus,Find the number of real solutions of the equat...,"Since $-1 \le \sin x \le 1,$ all solutions mus...",63
153612,Exp153612,Precalculus,"Let $A,$ $B,$ $C$ be the angles of a triangle....",We can expand the determinant as follows:\n\be...,0
153613,Exp153613,Precalculus,"Let $G$ be the centroid of triangle $ABC,$ and...","Let $\mathbf{a}$ denote $\overrightarrow{A},$ ...",3
153614,Exp153614,Precalculus,If angle $A$ lies in the second quadrant and $...,"Since angle $A$ lies in the second quadrant, $...",-\frac{\sqrt{7


In [8]:
system_prompt = """Your are a high school student appearing for your math exam.
You will be given a math problem to solve, and your job is to write the detailed step by step solution, with good mathematical reasoning and using python code for calculations, simplifications, solving equations, etc.

Format help and instructions:
- Problem has been given enclosed in <math_problem></math_problem> tags.
- Every step must be written within <start_of_step><end_of_step> tags.
- Code needs to be always written inside <code_block></code_block> tags - No backticks or triple backticks.
- Code output will be given in <code_output_block></code_output_block> tags

Every calculation should be performed using writing supporting python code."""

In [9]:
import signal
import functools
from time import sleep

class TimeoutExpired(Exception):
  """Custom exception for timeout"""
  pass

def timeout(seconds=10, error_message='Timeout'):
  """Decorator to timeout a function after a certain number of seconds.

  Args:
      seconds: The maximum number of seconds allowed for the function to run.
      error_message: The message to raise in case of timeout.

  Returns:
      The result of the decorated function if it finishes within the timeout,
      otherwise raises TimeoutExpired with the provided error message.
  """

  def decorator(func):
    @functools.wraps(func)  # Preserve function metadata
    def wrapper(*args, **kwargs):
      def _handle_timeout(signum, frame):
        raise TimeoutExpired(error_message)

      # Set the signal handler and a timer
      signal.signal(signal.SIGALRM, _handle_timeout)
      signal.alarm(seconds)

      try:
        result = func(*args, **kwargs)
      finally:
        # Cancel the alarm if the function finishes before timeout
        signal.alarm(0)
      return result
    return wrapper
  return decorator

In [10]:
@timeout(seconds=60, error_message="Code took too long!")
def run_code_v2(code:str) -> Tuple[str, Optional[str], Optional[str]]:
    with redirect_stdout(StringIO()) as f:
        try:
            exec(code)
            error_name = None
            error_msg = None
        except Exception as e:
            error_name = e.__class__.__name__
            error_msg = e
    return f.getvalue().strip(), error_name, error_msg

In [11]:
def get_code(output_str:str) -> str:
    return re.search(r"<code_block>(.*?)</code_block>", output_str, re.DOTALL).group(1).strip()

In [12]:
def get_final_answer(output_str:str) -> str:
    return re.search(r"<final_answer>(.*?)</final_answer>", output_str, re.DOTALL).group(1).strip()

In [13]:
def post_processing_v2(raw_output:str, input_text:str, end_reason:str):
    error_name, error_msg = None, None
    
    if end_reason == '</final_answer>':
        try:
            final_answer = get_final_answer(raw_output)
        except AttributeError:
            error_name = "FABlockSyntanxError"
            error_msg = None
        input_text += raw_output
    else:
        final_answer = None
    
    if end_reason == '<code_output_block>':
        # Get code block
        try:
            code = get_code(raw_output)
            # Runt the code
            code_output, error_name, error_msg = run_code_v2(code)
            input_text += f"{raw_output}{code_output}</code_output_block>\n"
        except AttributeError:
            error_name = "CodeBlockSyntanxError"
            error_msg = None
            input_text += raw_output
        
    elif end_reason == "<end_of_step>":
        input_text += f"{raw_output}\n"
        
    return input_text, final_answer, error_name, error_msg

In [14]:
def process_raw_output(all_outputs:list, main_df:pd.DataFrame) -> pd.DataFrame:
    exp_ids = []
    old_prompts = []
    responses = []
    new_prompts = []
    final_answers = []
    error_names = []
    error_msgs = []

    for idx in tqdm(range(len(all_outputs))):

        exp_id = main_df.ExpID[idx]
        old_prompt = all_inputs[idx]

        for raw_output_details in all_outputs[idx].outputs:

            assert old_prompt == all_outputs[idx].prompt

            input_text, final_answer, error_name, error_msg = post_processing_v2(raw_output_details.text, all_outputs[idx].prompt, raw_output_details.stop_reason)

            exp_ids.append(exp_id)
            old_prompts.append(all_outputs[idx].prompt)
            responses.append(raw_output_details.text)
            new_prompts.append(input_text)
            error_names.append(error_name)
            error_msgs.append(error_msg)
            final_answers.append(final_answer)
    return pd.DataFrame({
        "ExpID": exp_ids,
        "Old_Prompt": old_prompts,
        "Response": responses,
        "Current_Prompt": new_prompts,
        "Final_Answer": final_answers,
        "Error": error_names,
        "Error_Msg": error_msgs
    })
        

In [15]:
def save_as_jsonl(df:pd.DataFrame, cal_round:int):
    
    df['ExpID'] = df['ExpID'].astype(str)
    df['Old_Prompt'] = df['Old_Prompt'].astype(str)
    df['Response'] = df['Response'].astype(str)
    df['Current_Prompt'] = df['Current_Prompt'].astype(str)
    df['Final_Answer'] = df['Final_Answer'].progress_apply(lambda x: None if x is None else str(x))
    df['Error'] = df['Error'].progress_apply(lambda x: None if x is None else str(x))
    df['Error_Msg'] = df['Error_Msg'].progress_apply(lambda x: None if x is None else str(x))
    
    jsonl_data = []
    for _, current_row in tqdm(df.iterrows(), total=len(df)):

        data = {
            "ExpID": current_row['ExpID'],
            "Old_Prompt": current_row['Old_Prompt'],
            "Response": current_row['Response'],
            "Current_Prompt": current_row['Current_Prompt'],
            "Final_Answer": current_row['Final_Answer'],
            "Error": current_row['Error'],
            "Error_Msg": current_row['Error_Msg']
        }

        jsonl_data.append(data)
    
    with jsonlines.open(f"MATH_Inf_Outputs/Round{cal_round:0>2}.jsonl", 'w') as jlf:
        jlf.write_all(jsonl_data)

In [16]:
final_answers = {}
for exp_id, fa in tqdm(df[['ExpID', 'FinalAnswer']].values):
    final_answers[exp_id] = {'ofa': str(fa)}

  0%|          | 0/153616 [00:00<?, ?it/s]

In [17]:
def get_counters(final_answers:dict) -> Tuple[int, int, int]:
    
    correct_counter = 0
    wrong_counter = 0
    top3_counter = 0
    
    for exp_id in final_answers:
        ofa = final_answers[exp_id]['ofa']
        fa = final_answers[exp_id]['fa']
        top3 = final_answers[exp_id]['top3']
        
        if ofa == fa:
            correct_counter += 1
        else:
            wrong_counter += 1
        
        if ofa in top3:
            top3_counter += 1
    
    return correct_counter, wrong_counter, top3_counter

In [ ]:
problem_col = "Problem"
expid_col = "ExpID"
sampling_params = SamplingParams(
    n=5,
    temperature=0.2,
    top_p=1,
    top_k=100,
    max_tokens=1024,
    skip_special_tokens=False,
    spaces_between_special_tokens=False,
    stop=['<code_output_block>', '</final_answer>'],
    include_stop_str_in_output=True
)
total_threads = 0

# First Round
all_inputs = []

for prob in tqdm(df[problem_col]):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'problem', 'content': prob}
    ]

    all_inputs.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).replace(tokenizer.bos_token, ''))

# Warmup
llm.generate(all_inputs[0], sampling_params)

cal_round = 1
sampling_params.n = 5
while True:
    
    print(f"\n\n{'==' * 50}\n\nStarting Round {cal_round:0>2} : {len(all_inputs)} input prompts to process!\n")
    
    save_fname = f"MATH_Inf_Outputs/Round{cal_round:0>2}.jsonl"
    
    if os.path.exists(save_fname):
        df = pd.read_json(f"MATH_Inf_Outputs/Round{cal_round:0>2}.jsonl", lines=True)

    else:
        # Perform LLM Calls
        all_outputs = llm.generate(all_inputs, sampling_params)

        df = process_raw_output(all_outputs=all_outputs, main_df=df)

        try:
            df.to_json(save_fname, index=False, lines=True, orient='records')
        except OverflowError:
            save_as_jsonl(df, cal_round)

    # Collect Final Answers if any
    for _, current_row in df[df['Final_Answer'].notna()].iterrows():
        exp_id = current_row['ExpID']
        total_threads += 1

        if 'fas' not in final_answers[exp_id]:
            final_answers[exp_id]['fas'] = [current_row['Final_Answer']]
        else:
            final_answers[exp_id]['fas'].append(current_row['Final_Answer'])
        final_answers[exp_id]['top3'] = [str(x) for x, _  in Counter(final_answers[exp_id]['fas']).most_common(3)]
        final_answers[exp_id]['fas'] = str(Counter(final_answers[exp_id]['fas']).most_common(1)[0][0])

    
    correct_counter, wrong_counter, top3_counter = get_counters(final_answers)
    print(f'Stats after Round{cal_round:0>2}:')
    print(f'\tProblems Solved\t: {len(final_answers)} ({round(len(final_answers) / len(df) * 100, 2)}%)')
    print(f'\tTotal Threads\t: {total_threads}')
    print(f'\tCorrect Answer\t: {correct_counter} ({round(correct_counter / len(final_answers) * 100, 2)}%)')
    print(f'\tIncorrect Answer\t: {wrong_counter} ({round(wrong_counter / len(final_answers) * 100, 2)}%)')
    print(f'\tTop3 Answer\t: {top3_counter} ({round(top3_counter / len(final_answers) * 100, 2)}%)')
        
    # New batch
    new_df = df[(df['Final_Answer'].isna()) & (df['Error'].isna())].copy(deep=True).reset_index(drop=True)
    all_inputs = new_df['Current_Prompt'].values.tolist()
    cal_round += 1
    
    sampling_params.n = 1

    if len(all_inputs) == 0:
        break
    
    if cal_round > 15:
        break

  0%|          | 0/153616 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.21s/it, Generation Speed: 345.61 toks/s]





Starting Round 01 : 153616 input prompts to process!



Processed prompts:   1%|          | 1514/153616 [09:35<17:07:31,  2.47it/s, Generation Speed: 2147.17 toks/s]

In [ ]:
# cal_round = 3
# final_answers = {}
# sampling_params = SamplingParams(
#     n=1,
#     temperature=0.2,
#     top_p=1,
#     top_k=100,
#     max_tokens=1024,
#     skip_special_tokens=False,
#     spaces_between_special_tokens=False,
#     stop=['<code_output_block>', '</final_answer>'],
#     include_stop_str_in_output=True
# ) 

# for idx in tqdm(range(1, cal_round+1)):
#     df = pd.read_json(f"MATH_Inf_Outputs/Round{cal_round:0>2}.jsonl", lines=True)
    
#     # Collect Final Answers if any
#     for _, current_row in df[df['Final_Answer'].notna()].iterrows():
#         exp_id = current_row['ExpID']

#         if exp_id not in final_answers:
#             final_answers[exp_id] = [current_row['Final_Answer']]
#         else:
#             final_answers[exp_id].append(current_row['Final_Answer'])

# print(f"{len(final_answers)} / {len(MATH_numeric_df)} have been answered so far!")

# # New batch
# new_df = df[(df['Final_Answer'].isna()) & (df['Error'].isna())].copy(deep=True).reset_index(drop=True)
# new_df = new_df.drop_duplicates(['Current_Prompt'], ignore_index=True)
# all_inputs = new_df['Current_Prompt'].values.tolist()
# cal_round += 1


In [ ]:
# problem_col = "problem"
# expid_col = "ExpID"
# sampling_params = SamplingParams(
#     n=1,
#     temperature=0.2,
#     top_p=1,
#     top_k=100,
#     max_tokens=1024,
#     skip_special_tokens=False,
#     spaces_between_special_tokens=False,
#     stop=['<end_of_step>', '<code_output_block>', '</final_answer>'],
#     include_stop_str_in_output=True
# )

# while True:
    
#     print(f"\n\n{'==' * 50}\n\nStarting Round {cal_round:0>2}:\t{len(all_inputs)} input prompts to process!\n")
    
#     # Perform LLM Calls
#     all_outputs = llm.generate(all_inputs, sampling_params)

#     df = process_raw_output(all_outputs=all_outputs, main_df=df)
    
#     try:
#         df.to_json(f"MATH_Inf_Outputs/Round{cal_round:0>2}.jsonl", index=False, lines=True, orient='records')
#     except OverflowError:
#         save_as_jsonl(df, cal_round)

#     # Collect Final Answers if any
#     for _, current_row in df[df['Final_Answer'].notna()].iterrows():
#         exp_id = current_row['ExpID']

#         if exp_id not in final_answers:
#             final_answers[exp_id] = [current_row['Final_Answer']]
#         else:
#             final_answers[exp_id].append(current_row['Final_Answer'])

#     # New batch
#     new_df = df[(df['Final_Answer'].isna()) & (df['Error'].isna())].copy(deep=True).reset_index(drop=True)
#     new_df = new_df.drop_duplicates(['Current_Prompt'], ignore_index=True)
#     all_inputs = new_df['Current_Prompt'].values.tolist()
#     cal_round += 1
    
#     if len(all_inputs) == 0:
#         break